# Download Dataverse File with Guestbook

In [1]:
from pathlib import Path
import json
import requests
import os
import pandas as pd

from utils import format_guestbook_template, guestbook_questions_table, validate_response, extract_signed_url

## Set Up Intial Variables
**A. Change the following varaibles:**
1. Dataset PID (DOI - see citation)
2. File ID (embeded in file link)

**B. Find your unique API key:**
1. sign in to Dataverse
2. click your name in the upper right hand corner
3. select API Token
4. create an API Token if it is your first time using the API

**C. Set up your environment file:**
1. copy `.env.example` and rename `.env`
2. replace with your Dataverse API Token

In [2]:
SERVER = 'https://datasets.lib.berkeley.edu'
API_TOKEN = os.environ.get("API_TOKEN")

# Update with the desired dataset DOI
DATASET_PID = "doi:10.60503/D3/35DFHU"

# Right click and copy link to desired file for downloading
# Find the fileId embedded in the URL
FILE_ID = '33226'

## Get file and Guestbook Metadata

Run the following cells to call the Dataverse API and get metdata about the guestbook. If you do not get a successful response, check your above variables.

In [10]:
# Get filename and confirm it matches the file you want to download
file_metadata_resp = requests.get(
    f"{SERVER}/api/files/{FILE_ID}/metadata",
    headers={"X-Dataverse-key": API_TOKEN},
    timeout=60,
)
file_metadata_resp.raise_for_status()
filename = file_metadata_resp.json()["label"]

filename

'2003_Business_Academic_QCQ.txt.gz'

In [11]:
# Get Guestbook ID applied to the dataset
guestbook_id_resp = requests.get(
    f"{SERVER}/api/datasets/:persistentId/",
    params = {"persistentId": DATASET_PID},
    headers = {"X-dataverse-key": API_TOKEN},
    timeout=60,
)
guestbook_id_resp.raise_for_status()
dataset = guestbook_id_resp.json()["data"]
guestbook_id = dataset.get("guestbookId")

# Get metadata for guestbook
guestbook_formatted_resp = requests.get(
    f"{SERVER}/api/guestbooks/{guestbook_id}/",
    headers = {"X-dataverse-key": API_TOKEN},
    timeout=60
)
guestbook_formatted_resp.raise_for_status()
guestbook_json = guestbook_formatted_resp.json()["data"]

# Display guestbook response
guestbook_json

{'id': 10,
 'name': 'Historical Business Files',
 'enabled': True,
 'emailRequired': True,
 'nameRequired': True,
 'institutionRequired': False,
 'positionRequired': False,
 'customQuestions': [{'id': 20,
   'question': 'I will NOT compile, enhance, verify, supplement, add to/delete from mailing lists, geographic/trade, business, and classified directories, classified advertising, or compilations which are sold, rented, published, furnished or provided to a third party.',
   'required': True,
   'displayOrder': 0,
   'type': 'options',
   'hidden': False,
   'optionValues': [{'id': 46, 'value': 'I agree', 'displayOrder': 0},
    {'id': 36, 'value': 'I do not agree', 'displayOrder': 1}]},
  {'id': 21,
   'question': 'I will NOT make the Licensed Data or any portion available in an online environment except by a secured and encrypted bulletin board service, tape-to-tape batch transmission, or remote job entry, or in secured internal network environments.',
   'required': True,
   'displa

## Format Guestbook JSON Response

Steps:
1. Run `guestbook_questions_table` to see any custom questions (if a blank dataframe is returned, there are not any custom questions)
2. Run `json_response_template` to get properly formatted guestbook response
3. Copy the JSON output from `json_response_template`, paste it into the next cell, and save it as `completed_guestbook_response` (see `guestbook_questions_table` for valid answers)
4. Validate your completed response against the `form_spec`

Functions in this section are imported from `format_guestbook.py`.

In [4]:
# Run cell to see custom questions (if any)
guestbook_questions_table(guestbook_json)

,id,required,type,allowed responses,question
0,20,True,options,I agree; I do not agree,"I will NOT compile, enhance, verify, supplemen..."
1,21,True,options,I agree; I do not agree,I will NOT make the Licensed Data or any porti...
2,22,True,options,I agree; I do not agree,I will NOT use telephone number information in...
3,26,True,options,I agree; I do not agree,"I will NOT disassemble, decompile, reverse eng..."
4,25,True,options,I agree; I do not agree,"I will NOT use the Licensed Data, either in wh..."
5,24,True,options,I agree; I do not agree,"I will NOT use the Licensed Data, either in w..."
6,27,True,options,I agree; I do not agree,"I will NOT use the Licensed Data, either in wh..."
7,23,True,options,I agree; I do not agree,I will NOT use or allow third parties to use t...


In [5]:
# Run cell to print template JSON response
form_spec, json_response_template = format_guestbook_template(guestbook_json)

print(json.dumps(json_response_template, indent=2))

{
  "name": "",
  "email": "",
  "institution": "",
  "position": "",
  "answers": [
    {
      "id": 20,
      "value": null
    },
    {
      "id": 21,
      "value": null
    },
    {
      "id": 22,
      "value": null
    },
    {
      "id": 26,
      "value": null
    },
    {
      "id": 25,
      "value": null
    },
    {
      "id": 24,
      "value": null
    },
    {
      "id": 27,
      "value": null
    },
    {
      "id": 23,
      "value": null
    }
  ]
}


In [12]:
# Copy template JSON response into cell and fill out with valid answers
completed_guestbook_response = {}

In [7]:
# Run cell after completing template to ensure it is valid
validate_response(form_spec, completed_guestbook_response)

No errors found. Proceed to submit the guestbook response.


## Post Guestbook to API and Download file

Run final cell to download file

In [8]:
# Set up variables for signed URL request
endpoint = f"{SERVER}/api/access/datafile/{FILE_ID}"
params = {
    "signed": "true",
}
payload = {
    "guestbookResponse": completed_guestbook_response
}

# Request signed URL for file download
signed_url_resp = requests.post(
    endpoint, 
    params=params, 
    json=payload,
    headers={"X-Dataverse-key": API_TOKEN, "Accept": "application/json"},
    timeout=120)
print("Signed URL Status:", signed_url_resp.status_code)
signed_url_resp.raise_for_status()
signed_url_json = signed_url_resp.json()
signed_download_url = extract_signed_url(signed_url_json)

# Make directory for downloaded file if it doesn't exist
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# Download file using signed URL, save to data directory, and display progress
with requests.get(signed_download_url, stream=True, timeout=(30, 3600)) as resp:
    print("Download status:", resp.status_code)
    resp.raise_for_status()

    output_path = data_dir / filename

    total = 0
    with output_path.open("wb") as f:
        for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
            if chunk:
                f.write(chunk)
                total += len(chunk)
                print(f"\rDownloaded {total / (1024 * 1024):.1f} MiB", 
                      end="", flush=True
                     )

print(f"\nSaved to {output_path}")

Signed URL Status: 200
Download status: 200
Downloaded 769.9 MiB
Saved to data\2003_Business_Academic_QCQ.txt.gz
